<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_1/notebooks/Unit_1_Statistics_Assignment_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 1 Statistics Assignment — Olig2⁺GSTπ⁺ Density in CON and DRUG

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## The scenario

A second study used the same colocalization pipeline on a different pair of markers. Forty images —
sections from control animals (**CON**) and drug-treated animals (**DRUG**) — were stained for
**Olig2**, found in every cell of the oligodendrocyte lineage, and **GSTπ**, which appears only once
an oligodendrocyte has **matured** and begun myelinating.

Olig2 therefore counts the whole lineage; GSTπ counts only the finished cells. A cell positive for
**both** is a *mature oligodendrocyte*.

The readout is **`Colocalized_per_mm2`** — the density of double-positive cells in each image.
The lab's question:

> **Does the drug change the density of mature oligodendrocytes?**

You have a column of 40 numbers and no idea yet what they look like. Three steps take you from that
column to a claim you could defend.

## How this notebook works

**Run every cell in order, from the top.** Click a cell and press **Shift + Enter**.

| | What to do |
|---|---|
| **Ordinary cells** | Just run them. You don't need to understand every line of code. |
| **⚙️ Try it** | Change the highlighted value, re-run the cell, and see what happens. |
| **💬 Discussion** | All four are at the **end**. Work through the steps first, then answer them. |

**Symbols:** $X$ = one score · $\sum X$ = sum of scores · $n$ = number of scores ·
$\bar{X}$ = sample mean · $s$ = sample standard deviation

**To submit:** answer all four Discussions, click **Runtime → Run all**, then
**File → Print → Save as PDF** and upload the PDF to Canvas.

---

## Step 1 — Load the data and look at it [1 pt]

Before computing a single statistic, **look at your data**. A **histogram** sorts the values into bins
and shows how many fall into each, which is the fastest way to see the shape of a distribution.

Both groups are plotted with the **same bins and the same axes**. That matters: different bin widths
would make two identical groups look different.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/refs/heads/main/Unit_1/data/data_2.csv"
images = pd.read_csv(DATA)

# Pull the two groups out into their own variables
con = images[images["Group"] == "CON"]["Colocalized_per_mm2"]
drug = images[images["Group"] == "DRUG"]["Colocalized_per_mm2"]

print("Loaded", len(images), "images —", len(con), "CON and", len(drug), "DRUG\n")
print(images.head())

In [ ]:
BINS = np.arange(0, 75, 5)   # ⚙️ Try it — bin edges. Widen or narrow them and re-run.

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, sharey=True)

for ax, (name, data, colour) in zip(axes, [("CON", con, "gray"),
                                           ("DRUG", drug, "tomato")]):
    ax.hist(data, bins=BINS, color=colour, edgecolor="black")
    ax.set_title(name + " (n = " + str(len(data)) + ")")
    ax.set_xlabel("Olig2+GSTπ+ cells per mm²")

axes[0].set_ylabel("Number of images")
plt.tight_layout()
plt.show()

# The same information as numbers: how many images fell into each bin
print("Bin edges:", BINS)
print("  CON: ", np.histogram(con, bins=BINS)[0])
print("  DRUG:", np.histogram(drug, bins=BINS)[0])

## Step 2 — What is central tendency? [1 pt]

**Central tendency** is the pull of a distribution towards a middle value — the single number you would
give if someone asked "what is a typical image worth?" Three different statistics claim to answer that,
and they answer it differently:

- **Mean** $\bar{X} = \frac{\sum X}{n}$ — the average, and the balance point of the distribution.
  It uses **every** value, so one extreme image drags it.
- **Median** — the middle value once the data are sorted. It only cares about *position* in the sorted
  order, so an extreme image barely moves it.
- **Mode** — the most common value. For a continuous measurement like density no two images have
  exactly the same value, so we take the **midpoint of the tallest histogram bar**.

In a perfectly symmetric distribution all three land in the same place. When they don't, the size and
direction of their disagreement tells you about the **shape** of the distribution — which is what
Discussion 1 asks you about.

In [ ]:
def mode_from_histogram(x, bins):
    "For continuous data the mode is the midpoint of the tallest bar."
    heights, edges = np.histogram(x, bins=bins)
    tallest = np.argmax(heights)                  # which bar is highest
    return (edges[tallest] + edges[tallest + 1]) / 2


central_tendency = pd.DataFrame({
    "Mean": [con.mean(), drug.mean()],
    "Median": [con.median(), drug.median()],
    "Mode": [mode_from_histogram(con, BINS), mode_from_histogram(drug, BINS)],
}, index=["CON", "DRUG"]).round(2)

print("Olig2+GSTπ+ cells per mm²\n")
print(central_tendency)
print("\nDRUG mean / CON mean =", round(drug.mean() / con.mean(), 2))

In [ ]:
# The three measures drawn on top of each histogram
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)

for ax, (name, data, colour) in zip(axes, [("CON", con, "gray"),
                                           ("DRUG", drug, "tomato")]):
    ax.hist(data, bins=BINS, color=colour, edgecolor="white")
    ax.axvline(mode_from_histogram(data, BINS), color="green", lw=2.5, label="Mode")
    ax.axvline(data.median(), color="blue", lw=2.5, label="Median")
    ax.axvline(data.mean(), color="red", lw=2.5, label="Mean")
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.show()

# The same three numbers, sorted smallest to largest, for each group
for name, data in [("CON ", con), ("DRUG", drug)]:
    values = {"mode": mode_from_histogram(data, BINS),
              "median": data.median(),
              "mean": data.mean()}
    in_order = sorted(values, key=values.get)
    print(name, "|", "  <  ".join(w + " " + str(round(values[w], 2)) for w in in_order))

## Step 3 — What is dispersion? [1 pt]

**Dispersion** is how spread out the values are — how far a typical image sits from the middle. Two
groups can share a mean and still be completely different, so a typical value reported without a
dispersion is only half a result.

**IQR (interquartile range)** — the range covered by the middle 50%. Sort the data, find the value a
quarter of the way up (**Q1**) and three quarters of the way up (**Q3**), and subtract:
$IQR = Q3 - Q1$. Because it throws away the top and bottom quarters before measuring, one extreme image
cannot move it.

**Standard deviation ($s$)** — roughly the typical distance between one score and the mean:

$$s = \sqrt{\frac{\sum (X - \bar{X})^2}{n - 1}}$$

It uses **every** value, so unlike the IQR it does respond to extremes. Our 20 images per group are a
**sample** — we want to describe the drug in general, not only these sections — so the bottom is
$n - 1$. That's `ddof=1` in `pandas`. The default, `ddof=0`, is the population formula and is the wrong
one here.

Because the two measures treat extremes differently, they can disagree about which group is more spread
out. When they do, that disagreement is telling you something.

In [ ]:
spread = pd.DataFrame({
    "Q1": [con.quantile(0.25), drug.quantile(0.25)],
    "Median": [con.median(), drug.median()],
    "Q3": [con.quantile(0.75), drug.quantile(0.75)],
    "IQR": [con.quantile(0.75) - con.quantile(0.25),
            drug.quantile(0.75) - drug.quantile(0.25)],
    "SD": [con.std(ddof=1), drug.std(ddof=1)],
    "Min": [con.min(), drug.min()],
    "Max": [con.max(), drug.max()],
}, index=["CON", "DRUG"]).round(2)

print(spread)

# A box plot shows the median and the IQR directly: the line is the median,
# the box spans Q1 to Q3, and every image is drawn on top as a dot.
plt.figure(figsize=(5.5, 4.2))
plt.boxplot([con, drug], widths=0.5)
for position, data in enumerate([con, drug], start=1):
    jitter = np.random.default_rng(0).normal(position, 0.06, len(data))
    plt.scatter(jitter, data, color="black", alpha=0.6, zorder=3)
plt.xticks([1, 2], ["CON", "DRUG"])
plt.title("Every image, with median and IQR")
plt.show()

---

# 💬 Discussions

All four are below. Double-click a cell to type your answer.

---

### 💬 Discussion 1 — Is the data skewed? [2 pt]

Is each group **skewed (positive or negative)** — and if so, in which direction? Use the mode, median and mean from Step 2 to
support your answer.

**Your answer:** *(double-click here to type)*

>

---

### 💬 Discussion 2 — Mean or median? [2 pt]

Given the shape you just identified, would you use the **mean** or the **median** to describe a typical
image?

**Your answer:** *(double-click here to type)*

>

---

### 💬 Discussion 3 — True or false? [3 pt]


**True or false?**

> *"The DRUG group has a **lower** median than CON."*

> *"The DRUG group has a **larger** IQR."*

> *"The CON group has a **smaller** Standard Deviation (SD)."*


If it's false, correct the statement by writing it down in below.

**Your answer:** *(double-click here to type)*

>


---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.